## Transformer Architecture Implementation

In this notebook, I'm going to build the standard Encoder-Decoder Transformer architecture. The goal of this notebook is to get hands-on experience with building and training the Transformer architecture.

For this task, I picked a small English-to-French translation dataset from Kaggle.

**Task:** Machine Translation

**Dataset:** https://www.kaggle.com/datasets/vishwjeetmanwar/english-french

## Dataset Preparation

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "./eng-fra.txt", sep="\t", header=None, usecols=[0, 1], names=["english", "french"]
)

df

,english,french
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
...,...,...
239184,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
239185,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...
239186,If someone who doesn't know your background sa...,Si quelqu'un qui ne connaît pas vos antécédent...
239187,It may be impossible to get a completely error...,Il est peut-être impossible d'obtenir un Corpu...


## Tokenization & Vocabulary

In [13]:
english_vocab = {"<pad>": 0, "<eos>": 1}
french_vocab = {"<pad>": 0, "<bos>": 1, "<eos>": 2}


def word_tokenize(text):
    PUNCTUATION_TO_STRIP = '.,!?"()[]{}<>:;/=_+*&^%$#@`~'

    dirty_tokens = text.lower().split()
    clean_tokens = []

    for token in dirty_tokens:
        clean_token = token.strip(PUNCTUATION_TO_STRIP)
        if clean_token:
            clean_tokens.append(clean_token)

    return clean_tokens


def forming_vocab(text, lang):
    tokenize_text = word_tokenize(text)

    if lang == "eng":
        for word in tokenize_text:
            if word not in english_vocab:
                english_vocab[word] = len(english_vocab)

    if lang == "fre":
        for word in tokenize_text:
            if word not in french_vocab:
                french_vocab[word] = len(french_vocab)


# applying
df["english"].apply(lambda text: forming_vocab(text, "eng"))
df["french"].apply(lambda text: forming_vocab(text, "fre"))

print("English vocab:", len(english_vocab))
print("French vocab:", len(french_vocab))

English vocab: 17598
French vocab: 34222


## Numericalization

In [14]:
def text_to_numbers(sent, vocab, lang):
    sent_tok = word_tokenize(sent)
    text_seq = []

    if lang == "eng":
        for word in sent_tok:
            if word in vocab:
                text_seq.append(vocab[word])

        text_seq.append(vocab["<eos>"])

    if lang == "fre":
        text_seq.append(vocab["<bos>"])

        for word in sent_tok:
            if word in vocab:
                text_seq.append(vocab[word])

        text_seq.append(vocab["<eos>"])

    return text_seq


english_sequences = df["english"].apply(
    lambda sen: text_to_numbers(sen, english_vocab, "eng")
)
french_sequences = df["french"].apply(
    lambda sen: text_to_numbers(sen, french_vocab, "fre")
)


def padding(max_len, num_sequence):
    LIST = []
    zeroes = [0] * (max_len - len(num_sequence))
    LIST = num_sequence + zeroes

    return LIST


def getMaxLength(seq):
    max_len = 0

    for s in seq.values:
        if len(s) > max_len:
            max_len = len(s)

    return max_len


print(f"Max length of English:", getMaxLength(english_sequences))
print(f"Max length of French:", getMaxLength(french_sequences))

Max length of English: 56
Max length of French: 68


## Sequence Preparation

In [15]:
from torch.utils.data import Dataset, DataLoader


class data_formation(Dataset):
    def __init__(self, english_sequences, french_sequences):
        self.english_sequences = english_sequences
        self.french_sequences = french_sequences

    def __len__(self):
        return self.english_sequences.shape[0]

    def __getitem__(self, index):
        english_sequence = padding(55, self.english_sequences[index])
        french_sequence = padding(68, self.french_sequences[index])

        return torch.tensor(english_sequence, dtype=torch.long), torch.tensor(
            french_sequence, dtype=torch.long
        )


# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    english_sequences.values,
    french_sequences.values,
    test_size=0.15,
    shuffle=True,
    random_state=42,
)

training_dataset = data_formation(X_train, y_train)
testing_dataset = data_formation(X_test, y_test)

training_loader = DataLoader(training_dataset, batch_size=2)
testing_loader = DataLoader(testing_dataset, batch_size=2)

In [16]:
for x, y in training_loader:
    print(x, end="\n")
    print(y)
    break

tensor([[  91,  985, 3161,  708,   63, 2376,  902,    1,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [ 107,  249,  753,   63,    1,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0]])
tensor([[   1, 3657,  136, 1518,  726,  853,  168,  481, 2082,    2,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
    

## Positional Encoding 

PE(pos, 2i) = sin(pos / 10000^(2i / d_model))

PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

In [1]:
# Positional Encoding matrix
import numpy as np
import torch


def get_Pencoding(
    max_seq_len: int, d_model: int, last_tok_index=None, pad: bool = False
):

    PE = np.zeros((max_seq_len, d_model))

    for pos in range(max_seq_len):
        for i in range(int(d_model / 2)):

            # pos / (10000 ** (2i/512))
            angle = pos * np.exp(-(2 * i / d_model) * np.log(10000.0))

            # get Sine
            PE[pos, 2 * i] = np.sin(angle)

            # get Cosine
            PE[pos, 2 * i + 1] = np.cos(angle)

    if pad:

        if not last_tok_index:
            raise ValueError(
                "Padding is required to identify the index of the last token in the sequence."
            )

        elif last_tok_index > max_seq_len:
            raise ValueError("last_tok_index must be <= max_seq_len")

        with_padding = np.pad(
            PE,
            ((0, max_seq_len - last_tok_index), (0, 0)),
            mode="constant",
            constant_values=0,
        )

        return with_padding

    else:
        return PE


class TransformerEmbedding(torch.nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()

        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=0
        )
        self.positional_encoding = get_Pencoding(max_seq_len, d_model)

    def forward(self, x):

        emb_matrix = self.embedding(x)
        self.positional_encoding
        Pos_tensor = torch.from_numpy(self.positional_encoding).to(
            device=emb_matrix.device, dtype=emb_matrix.dtype
        )

        return emb_matrix + Pos_tensor

# MHA

There are two ways to implement Multi-Head Attention (MHA): **Explicit Parameters** and **Merged Linear**.

* **Explicit Parameters:** This approach is easier to understand and closely follows the implementation described in the paper. We create separate, smaller weight matrices for each attention head: \(W^Q\), \(W^K\), and \(W^V\).

* **Merged Linear:** This approach is more efficient. Instead of creating separate matrices for each head, we create one large matrix for Q, K, and V, and then split the resulting output across the attention heads.

For better understanding, we will use **both approaches** in our implementation:

* **Encoder:** Explicit Parameters
* **Decoder:** Merged Linear


In [28]:
class EpMultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert (
            d_model % num_heads == 0
        ), f"Embedding dim {d_model} not divisible by {num_heads} heads"

        self.head_dim = d_model // num_heads

        # weight Matrices
        self.Wq = torch.nn.Parameter(torch.randn(num_heads, d_model, self.head_dim))
        self.Wk = torch.nn.Parameter(torch.randn(num_heads, d_model, self.head_dim))
        self.Wv = torch.nn.Parameter(torch.randn(num_heads, d_model, self.head_dim))

        # Output projection
        self.Wo = torch.nn.Parameter(torch.randn(num_heads, self.head_dim, d_model))

        # Layer Norm for the Encoder stabilization
        self.layer_norm = torch.nn.LayerNorm(d_model)

        # Initialize weights, Important for this type of implementation
        self._init_weights()

    def _init_weights(self):
        # Xavier/Glorot initialization is crucial for custom Parameter tensors
        torch.nn.init.xavier_uniform_(self.Wq)
        torch.nn.init.xavier_uniform_(self.Wk)
        torch.nn.init.xavier_uniform_(self.Wv)
        torch.nn.init.xavier_uniform_(self.Wo)

    def forward(self, x):

        # projections
        q_heads = torch.einsum("btd, hdk -> bhtk", x, self.Wq)
        k_heads = torch.einsum("btd, hdk -> bhtk", x, self.Wk)
        v_heads = torch.einsum("btd, hdk -> bhtk", x, self.Wv)

        # Attention scores
        As = torch.einsum("bhqd, bhkd -> bhqk", q_heads, k_heads) * (
            self.head_dim**-0.5
        )  # bhqk mean (batch, head, token, token)

        # Attention probabilities
        wei = torch.nn.functional.softmax(As, dim=-1)

        # Final context aggregation
        out = torch.einsum("bhqk, bhkd -> bhqd", wei, v_heads)

        #  Merge all attention heads
        final_output = torch.einsum("bhtk, hkd -> btd", out, self.Wo)

        return self.layer_norm(x + final_output)

## Feed-Forward Network (FFN)
FFN(x) = max(0, xW1 + b1 )W2 + b2

In [29]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, d_model, dropout_rate=0.2):
        super().__init__()

        self.layer_1 = torch.nn.Linear(d_model, d_model * 4)
        self.relu = torch.nn.ReLU()
        self.layer_2 = torch.nn.Linear(d_model * 4, d_model)

        self.layer_norm = torch.nn.LayerNorm(d_model)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x):

        out = self.layer_1(x)
        out = self.relu(out)
        out = self.layer_2(out)

        out = self.dropout(out)

        return self.layer_norm(x + out)

## Encoder Architecture

In [34]:
class Encoder(torch.nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, num_encoder_layers):
        super().__init__()

        self.num_encoder_layers = num_encoder_layers

        self.pos_encoding = TransformerEmbedding(vocab_size, max_seq_len, d_model)
        self.MHA = EpMultiHeadAttention(d_model, num_heads)
        self.FFNN = FeedForwardNetwork(d_model)

    def forward(self, x):

        x = self.pos_encoding(x)

        for _ in range(self.num_encoder_layers):

            x = self.MHA(x)
            x = self.FFNN(x)

        return x

In [35]:
vocab_size = 17598
d_model = 512
max_seq_len = 56
num_heads = 4
num_encoder_layers = 5

input_matrix = torch.randint(0, vocab_size, (2, max_seq_len), dtype=torch.long)

en = Encoder(vocab_size, max_seq_len, d_model, num_heads, num_encoder_layers)

encoder_output = en.forward(input_matrix)
encoder_output.shape

torch.Size([2, 56, 512])

## Decoder Architecture

**Masked Maulti head Attention**

In [6]:
class MaskedMultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len):
        super().__init__()

        assert (
            d_model % num_heads == 0
        ), f"Embedding dim {d_model} not divisible by {num_heads} heads"

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Q, K, V projections
        self.query = torch.nn.Linear(d_model, d_model)
        self.key = torch.nn.Linear(d_model, d_model)
        self.value = torch.nn.Linear(d_model, d_model)

        self.proj_out = torch.nn.Linear(d_model, d_model)

        self.mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()

    def forward(self, x):

        batch, tokens, dim = x.shape

        # Q, K, V -> (B, T, D)
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Split embeddings into multiple heads | (B, T, D) -> (B, T, H, HD) -> (B, H, T,  HD)
        q = q.view(batch, tokens, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, tokens, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, tokens, self.num_heads, self.head_dim).transpose(1, 2)

        As = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (
            self.head_dim**-0.5
        )  # (batch, head, token, token)

        # Mask to prevident model to see feature tokens
        wei = As.masked_fill(self.mask, float("-inf"))

        wei = torch.nn.functional.softmax(wei, dim=-1)  # On Rows
        out = torch.einsum("bhqk, bhkd -> bhqd", wei, v).transpose(
            1, 2
        )  # (B, H, T, HD) -> (B, T, H, HD)

        out = out.reshape(batch, tokens, dim)  # (B, T, D)

        # merging all head
        final_output = self.proj_out(out)

        return final_output  # Norm-Not added

**Multi-Head Attention with Merged QKV Linear Projection**

In [ ]:
class MlMultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert (
            d_model % num_heads == 0
        ), f"Embedding dim {d_model} not divisible by {num_heads} heads"

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Q, K, V projections
        self.query = torch.nn.Linear(d_model, d_model)
        self.key = torch.nn.Linear(d_model, d_model)
        self.value = torch.nn.Linear(d_model, d_model)

        self.proj_out = torch.nn.Linear(d_model, d_model)
        self.layer_norm = torch.nn.LayerNorm(d_model)

    def forward(self, decoder_input, encoder_input):

        EB, ET, ED = encoder_input.shape
        DB, DT, DD = decoder_input.shape

        # Q, K, V projection -> (B, T, D)
        q = self.query(decoder_input)
        k = self.key(encoder_input)
        v = self.value(encoder_input)

        # Split embeddings into multiple heads | (B, T, D) -> (B, T, H, HD) -> (B, H, T,  HD)
        q = q.view(DB, DT, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(EB, ET, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(EB, ET, self.num_heads, self.head_dim).transpose(1, 2)

        As = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (
            self.head_dim**-0.5
        )  # (B, H, T, T)

        wei = torch.nn.functional.softmax(As, dim=-1)  # Applying on Rows
        out = torch.einsum("bhqk, bhkd  -> bhqd", wei, v).transpose(
            1, 2
        )  # (B, DT, H, HD) → (B, DT, D)

        out = out.reshape(DB, DT, DD)  # Back to Decoder Shape

        # merging all head
        final_output = self.proj_out(out)

        return self.layer_norm(decoder_input + final_output)

In [8]:
mmha = MaskedMultiHeadAttention(d_model, num_heads, max_seq_len)
decoder_input = torch.randn((3, max_seq_len, d_model))
decoder_input.shape
mmha.forward(decoder_input)

tensor([[[-0.0336, -0.1429, -0.4553,  ..., -0.1560,  0.2918, -0.2885],
         [-0.1091,  0.2367, -0.3792,  ..., -0.3446,  0.0822, -0.5914],
         [-0.0902, -0.0349, -0.3508,  ..., -0.3921,  0.2734, -0.3120],
         ...,
         [-0.0305,  0.0128, -0.0973,  ..., -0.0781,  0.0791, -0.0125],
         [-0.0212,  0.0118, -0.1080,  ..., -0.0930,  0.0623, -0.0119],
         [ 0.0163,  0.0061, -0.0856,  ..., -0.0454,  0.0623, -0.0306]],

        [[-0.0972,  0.2648, -0.2619,  ..., -0.5530, -0.0350,  0.5183],
         [ 0.3348,  0.0061,  0.1494,  ..., -0.1333, -0.1522,  0.4796],
         [-0.0087,  0.0742, -0.0070,  ..., -0.1953,  0.0280,  0.3754],
         ...,
         [-0.0273, -0.0533, -0.0455,  ...,  0.0110,  0.1078,  0.0246],
         [-0.0207, -0.0165, -0.0170,  ...,  0.0153,  0.1237, -0.0016],
         [-0.0491, -0.0543, -0.0582,  ..., -0.0174,  0.1154,  0.0034]],

        [[-0.2695, -0.2659,  0.4942,  ..., -0.2264,  0.1237,  0.1903],
         [-0.2963, -0.2728,  0.3863,  ..., -0

## Encoder–Decoder Model

## Training Loop

## Inference / Translation

## BLEU Evaluation